In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class BakSneppenModel:
    """
    Simulates the 1D Bak-Sneppen model on a ring of L species.
    """
    def __init__(self, L=100, seed=None):
        self.L = L
        if seed is not None:
            np.random.seed(seed)
        self.fitness = np.random.rand(L)
        
    def step(self):
        """
        Executes a single step of Bak-Sneppen dynamics:
        Identifies minimum fitness site and mutates it along with its two circular neighbors.
        """
        min_idx = np.argmin(self.fitness)
        min_val = self.fitness[min_idx]
        
        left_idx = (min_idx - 1) % self.L
        right_idx = (min_idx + 1) % self.L
        
        self.fitness[min_idx] = np.random.rand()
        self.fitness[left_idx] = np.random.rand()
        self.fitness[right_idx] = np.random.rand()
        
        return min_idx, min_val

    def run_simulation(self, n_iterations=8000):
        min_indices = np.zeros(n_iterations, dtype=int)
        min_fitnesses = np.zeros(n_iterations)
        mean_fitnesses = np.zeros(n_iterations)
        
        for t in range(n_iterations):
            min_idx, min_val = self.step()
            min_indices[t] = min_idx
            min_fitnesses[t] = min_val
            mean_fitnesses[t] = np.mean(self.fitness)
            
        return {
            'min_indices': min_indices,
            'min_fitnesses': min_fitnesses,
            'mean_fitnesses': mean_fitnesses,
            'final_fitness': self.fitness.copy()
        }


def generate_figure_1_spacetime_patterns(output_path='bak_sneppen_spacetime_patterns.png'):
    """
    Figure 1: Space-Time Diagrams for Two Independent Runs (KR2 & KR3).
    """
    L = 100
    n_iter = 8000
    
    # Run 1
    model1 = BakSneppenModel(L=L, seed=101)
    run1 = model1.run_simulation(n_iterations=n_iter)
    
    # Run 2
    model2 = BakSneppenModel(L=L, seed=202)
    run2 = model2.run_simulation(n_iterations=n_iter)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 8), sharey=True)
    
    img1 = np.zeros((n_iter, L))
    for t in range(n_iter):
        img1[t, run1['min_indices'][t]] = 1

    img2 = np.zeros((n_iter, L))
    for t in range(n_iter):
        img2[t, run2['min_indices'][t]] = 1

    ax1.imshow(img1, aspect='auto', cmap='binary', origin='upper', interpolation='none')
    ax1.set_title('Space-Time Pattern of Extinction Sites (Run 1)\nSeed=101, L=100, 8,000 Iterations', fontsize=12, pad=10)
    ax1.set_xlabel('Species Location Index $i$ (Ring Topology)', fontsize=11)
    ax1.set_ylabel('Iteration Step $t$', fontsize=11)

    ax2.imshow(img2, aspect='auto', cmap='binary', origin='upper', interpolation='none')
    ax2.set_title('Space-Time Pattern of Extinction Sites (Run 2)\nSeed=202, L=100, 8,000 Iterations', fontsize=12, pad=10)
    ax2.set_xlabel('Species Location Index $i$ (Ring Topology)', fontsize=11)

    plt.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)


def generate_figure_2_steady_state_convergence(output_path='bak_sneppen_steady_state_convergence.png'):
    """
    Figure 2: Steady-State Convergence & Critical Threshold Analysis (KR4).
    """
    L = 100
    n_iter = 20000
    n_runs = 30
    
    mean_fitness_runs = np.zeros((n_runs, n_iter))
    min_fitness_sample = None
    
    for r in range(n_runs):
        model = BakSneppenModel(L=L, seed=1000 + r)
        res = model.run_simulation(n_iterations=n_iter)
        mean_fitness_runs[r, :] = res['mean_fitnesses']
        if r == 0:
            min_fitness_sample = res['min_fitnesses']

    ensemble_mean = np.mean(mean_fitness_runs, axis=0)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

    # Plot 1: Ensemble Average & Individual Run Mean Fitness
    ax1.plot(mean_fitness_runs[0, :], color='lightblue', alpha=0.5, label='Single Run Mean $\\langle f \\rangle(t)$')
    for r in range(1, 5):
        ax1.plot(mean_fitness_runs[r, :], color='lightblue', alpha=0.3)
    ax1.plot(ensemble_mean, color='navy', linewidth=2, label=f'Ensemble Average (N={n_runs} Runs)')
    
    # Theoretical asymptote <f>_steady = (B_c + 1)/2 = (0.667 + 1)/2 = 0.8335
    ax1.axhline(y=0.8335, color='crimson', linestyle='--', linewidth=1.8, label=r'Theoretical Steady-State $\langle f \rangle \approx 0.833$ ($B_c \approx 0.667$)')
    ax1.axvline(x=2000, color='gray', linestyle=':', linewidth=1.5, label='Transient Boundary ($t \\approx 2,000$)')
    
    ax1.set_ylabel('Mean System Fitness $\\langle f \\rangle$', fontsize=11)
    ax1.set_title('Bak-Sneppen Model Convergence to Dynamical Steady State ($L=100$)', fontsize=13, pad=10)
    ax1.legend(loc='lower right', frameon=True)
    ax1.grid(True, linestyle=':', alpha=0.6)

    # Plot 2: Minimum Fitness Time Series
    ax2.plot(min_fitness_sample, color='darkgreen', alpha=0.4, linewidth=0.6, label='Minimum Fitness $f_{min}(t)$')
    window = 100
    smoothed_min = np.convolve(min_fitness_sample, np.ones(window)/window, mode='valid')
    ax2.plot(np.arange(window-1, n_iter), smoothed_min, color='darkgreen', linewidth=1.8, label='Smoothed $f_{min}(t)$ (100-step Moving Avg)')
    ax2.axhline(y=0.667, color='crimson', linestyle='--', linewidth=1.8, label=r'Critical Threshold $B_c \approx 0.667$')
    ax2.axvline(x=2000, color='gray', linestyle=':', linewidth=1.5)

    ax2.set_xlabel('Iteration Step $t$', fontsize=11)
    ax2.set_ylabel('Minimum Fitness $f_{min}$', fontsize=11)
    ax2.set_ylim(-0.02, 1.02)
    ax2.legend(loc='lower right', frameon=True)
    ax2.grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)


def generate_figure_3_additional_analysis(output_path='bak_sneppen_additional_analysis.png'):
    """
    Figure 3: Additional Quantitative Analyses (KR5).
    - Stationary Fitness Distribution P(f)
    - Spatial Jump Distance Distribution C(x)
    - Avalanche Size Distribution P(s)
    """
    L = 100
    n_iter = 50000
    
    model = BakSneppenModel(L=L, seed=42)
    
    min_indices = np.zeros(n_iter, dtype=int)
    min_fitnesses = np.zeros(n_iter)
    all_fitness_samples = []
    
    for t in range(n_iter):
        idx, val = model.step()
        min_indices[t] = idx
        min_fitnesses[t] = val
        if t >= 10000 and t % 50 == 0:
            all_fitness_samples.extend(model.fitness.copy())
            
    all_fitness_samples = np.array(all_fitness_samples)
    
    # 1. Spatial jump distance distribution C(x)
    steady_indices = min_indices[10000:]
    jumps = np.abs(np.diff(steady_indices))
    jumps = np.minimum(jumps, L - jumps)
    jumps = jumps[jumps > 0]
    
    x_vals, x_counts = np.unique(jumps, return_counts=True)
    c_x = x_counts / np.sum(x_counts)
    
    # 2. Avalanche size distribution P(s)
    B_0 = 0.65
    steady_min_fit = min_fitnesses[10000:]
    
    avalanches = []
    current_size = 0
    for f in steady_min_fit:
        if f < B_0:
            current_size += 1
        else:
            if current_size > 0:
                avalanches.append(current_size)
                current_size = 0
    if current_size > 0:
        avalanches.append(current_size)
        
    avalanches = np.array(avalanches)
    s_vals, s_counts = np.unique(avalanches, return_counts=True)
    p_s = s_counts / np.sum(s_counts)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

    # Panel A: P(f)
    ax1.hist(all_fitness_samples, bins=40, density=True, color='skyblue', edgecolor='navy', alpha=0.7)
    ax1.axvline(x=0.667, color='crimson', linestyle='--', linewidth=2, label=r'Critical Threshold $B_c \approx 0.667$')
    ax1.set_title('A. Stationary Fitness Distribution $P(f)$', fontsize=12, pad=10)
    ax1.set_xlabel('Fitness $f$', fontsize=11)
    ax1.set_ylabel('Probability Density $P(f)$', fontsize=11)
    ax1.legend(loc='upper left', frameon=True)
    ax1.grid(True, linestyle=':', alpha=0.6)

    # Panel B: C(x) Log-Log Plot
    ax2.loglog(x_vals, c_x, 'o-', color='purple', linewidth=1.5, markersize=5, label='Empirical $C(x)$')
    if len(x_vals) > 2:
        fit_params = np.polyfit(np.log(x_vals[:15]), np.log(c_x[:15]), 1)
        fit_line = np.exp(fit_params[1]) * (x_vals ** fit_params[0])
        ax2.loglog(x_vals, fit_line, '--', color='orange', label=f'Power-law Fit $\\propto x^{{{fit_params[0]:.2f}}}$')
    
    ax2.set_title('B. Spatial Jump Distance Distribution $C(x)$', fontsize=12, pad=10)
    ax2.set_xlabel('Spatial Jump Distance $x$', fontsize=11)
    ax2.set_ylabel('Probability $C(x)$', fontsize=11)
    ax2.legend(loc='lower left', frameon=True)
    ax2.grid(True, which='both', linestyle=':', alpha=0.6)

    # Panel C: P(s) Log-Log Plot
    ax3.loglog(s_vals, p_s, 's-', color='darkgreen', linewidth=1.5, markersize=4, label='Avalanche Distribution $P(s)$')
    if len(s_vals) > 2:
        s_fit_idx = (s_vals >= 1) & (s_vals <= 50)
        if np.sum(s_fit_idx) > 2:
            s_fit = np.polyfit(np.log(s_vals[s_fit_idx]), np.log(p_s[s_fit_idx]), 1)
            s_line = np.exp(s_fit[1]) * (s_vals ** s_fit[0])
            ax3.loglog(s_vals, s_line, '--', color='red', label=f'Power-law Fit $\\propto s^{{{s_fit[0]:.2f}}}$')
            
    ax3.set_title(f'C. Coevolutionary Avalanche Distribution $P(s)$ ($B_0={B_0}$)', fontsize=12, pad=10)
    ax3.set_xlabel('Avalanche Size $s$', fontsize=11)
    ax3.set_ylabel('Probability $P(s)$', fontsize=11)
    ax3.legend(loc='lower left', frameon=True)
    ax3.grid(True, which='both', linestyle=':', alpha=0.6)

    plt.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)


def main():
    print("=======================================================================")
    print(" Generating All Visualizations for Bak-Sneppen Evolutionary Model       ")
    print("=======================================================================")
    
    generate_figure_1_spacetime_patterns('bak_sneppen_spacetime_patterns.png')
    generate_figure_2_steady_state_convergence('bak_sneppen_steady_state_convergence.png')
    generate_figure_3_additional_analysis('bak_sneppen_additional_analysis.png')
    
    print("\nAll figures generated successfully!")


if __name__ == '__main__':
    main()


 Generating All Visualizations for Bak-Sneppen Evolutionary Model       

All figures generated successfully!
